# 6장 1강: CV 기반 하이퍼파라미터 튜닝과 모델 탐색

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 하이퍼파라미터 탐색과 교차검증을 결합해 최적 모델을 찾습니다.

- GridSearchCV와 RandomizedSearchCV의 탐색 방식과 비용 차이를 비교합니다.
- Optuna의 TPE 탐색을 이용해 초기 탐색 이후 이전 trial 결과를 활용하는 탐색을 수행합니다.
- 교차검증 평균 점수와 표준편차를 함께 확인합니다.
- 탐색 시간까지 고려하여 최종 후보를 선정합니다.
- 처음부터 분리해 둔 hold-out test set은 최종 모델을 정한 뒤 한 번만 사용합니다.
- 최종 모델과 메타데이터를 저장합니다.

> 이 실습에서는 6장 1강 교안에서 다룬 Grid Search, Random Search, Optuna, CV 평균·표준편차, 탐색 비용, hold-out test, 모델 저장 내용만 사용합니다.


## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- scipy
- Optuna
- joblib
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

하이퍼파라미터 탐색에 Test 데이터를 사용하면 안 됩니다.

따라서 먼저 전체 데이터를 **Train / hold-out Test**로 분리하고,

- 하이퍼파라미터 탐색과 교차검증 → Train 데이터만 사용
- 최종 성능 확인 → 모든 선택이 끝난 뒤 hold-out Test를 한 번만 사용

하는 흐름으로 진행합니다.

In [1]:
import os
import json
import time
import joblib
import optuna
import pandas as pd

from datetime import datetime
from scipy.stats import randint

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Hold-out Test 크기:", X_test.shape)

c:\ml2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train 크기: (1168, 12)
Hold-out Test 크기: (292, 12)


---

# 필수 1. Grid Search와 Random Search 비교

## 배경

GridSearchCV는 지정한 모든 조합을 빠짐없이 평가하고, RandomizedSearchCV는 정해진 횟수만큼 조합을 무작위로 선택합니다.

이번 실습에서는 같은 RandomForestRegressor를 대상으로 두 탐색 전략을 비교합니다.

## 문제 1. GridSearchCV를 수행하세요.

### 요구사항

1. `RandomForestRegressor(random_state=42, n_jobs=-1)`를 사용합니다.
2. 다음 후보를 사용합니다.

```python
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}
```

3. `cv=5`를 사용합니다.
4. `scoring="r2"`를 사용합니다.
5. 탐색 시간을 측정합니다.
6. `best_params_`와 `best_score_`를 출력합니다.
7. `cv_results_`에서 최적 조합의 `mean_test_score`, `std_test_score`를 확인합니다.

### 확인 포인트

- 총 9개 조합을 모두 평가하는가?
- 5-Fold이므로 각 조합이 5번 평가되는 구조를 이해했는가?
- 평균 점수뿐 아니라 표준편차도 확인했는가?

In [2]:
# TODO
# GridSearchCV를 수행하세요.
import time
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# 1. 기본 모델 설정
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# 2. 탐색 파라미터 그리드 정의 (총 3 * 3 = 9개 조합)
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}

# 3~4. GridSearchCV 객체 생성 (cv=5, scoring="r2")
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=grid_params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

# 5. 탐색 시간 측정 및 학습 진행 (Train 데이터만 사용)
start_time = time.time()
grid_search.fit(X_train, y_train)
grid_search_time = time.time() - start_time

# 6. 최적 파라미터 및 최고 점수 확인
print("=== GridSearchCV 탐색 결과 ===")
print(f"탐색 소요 시간: {grid_search_time:.2f}초")
print(f"최적 파라미터 (best_params_): {grid_search.best_params_}")
print(f"최고 CV R² 점수 (best_score_): {grid_search.best_score_:.4f}")

# 7. cv_results_에서 최적 조합의 평균 점수와 표준편차 확인
cv_results = pd.DataFrame(grid_search.cv_results_)
best_idx = grid_search.best_index_
best_mean = cv_results.loc[best_idx, "mean_test_score"]
best_std = cv_results.loc[best_idx, "std_test_score"]

print(f"\n[최적 조합 상세 통계]")
print(f"Mean CV R² : {best_mean:.4f}")
print(f"Std CV R²  : {best_std:.4f}")

# 전체 9개 조합 요약 테이블 출력
summary_cols = ["param_max_depth", "param_n_estimators", "mean_test_score", "std_test_score", "rank_test_score"]
display(cv_results[summary_cols].sort_values(by="rank_test_score").reset_index(drop=True))

=== GridSearchCV 탐색 결과 ===
탐색 소요 시간: 4.66초
최적 파라미터 (best_params_): {'max_depth': 12, 'n_estimators': 200}
최고 CV R² 점수 (best_score_): 0.8206

[최적 조합 상세 통계]
Mean CV R² : 0.8206
Std CV R²  : 0.0644


,param_max_depth,param_n_estimators,mean_test_score,std_test_score,rank_test_score
0,12,200,0.820618,0.064425,1
1,12,100,0.820348,0.065524,2
2,12,300,0.820182,0.065787,3
3,8,300,0.818850,0.063274,4
4,8,200,0.817849,0.064449,5
5,8,100,0.817761,0.065969,6
6,4,300,0.780580,0.050523,7
7,4,200,0.779586,0.049544,8
8,4,100,0.779369,0.051308,9


###
- 최고 CV 평균 0.82, max_depth와 n_estimators의 조합으로 5개의 검증 R²을 평균한 값
###
- CV 표준편차 0.06, 최적 조합 Fold별로 검증 점수가 평균 주변에서 얼마나 달라졌는지를 나타냄

### Q1. Grid Search의 가장 큰 장점과 한계는 무엇인가요?

**답변:**  
- 장점: 지정한 조합을 빠짐없이 평가하여 성능이 좋은 조합을 찾을 수 있다. 
- 단점: 후보 수가 많아지면 시간이 오래 걸린다.

## 문제 2. RandomizedSearchCV를 수행하세요.

### 요구사항

1. 같은 RandomForestRegressor를 사용합니다.
2. 다음 탐색 공간을 사용합니다.

```python
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}
```

3. `n_iter=9`를 사용합니다.
4. `cv=5`, `scoring="r2"`를 사용합니다.
5. `random_state=42`를 사용합니다.
6. 탐색 시간을 측정합니다.
7. 최적 파라미터, CV 평균 점수, 표준편차를 확인합니다.

### 확인 포인트

- 전체 조합을 모두 보는 것이 아니라 9번만 무작위로 탐색하는가?
- `n_iter`가 탐색 예산을 직접 정한다는 점을 이해했는가?

In [3]:
# TODO
# RandomizedSearchCV를 수행하세요.
import time
import pandas as pd
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

# 1. 동일한 기본 RandomForestRegressor 모델 설정
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# 2. 연속적/이산적 분포를 갖는 탐색 공간 정의
# randint(low, high)는 [low, high - 1] 범위이므로 100~300, 4~12를 포함
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}

# 3~5. RandomizedSearchCV 객체 생성 (n_iter=9, cv=5, scoring="r2", random_state=42)
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=random_params,
    n_iter=9,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

# 6. 탐색 시간 측정 및 학습 진행 (Train 데이터만 사용)
start_time = time.time()
random_search.fit(X_train, y_train)
random_search_time = time.time() - start_time

# 7. 최적 파라미터, 최고 점수 및 표준편차 확인
cv_results_rand = pd.DataFrame(random_search.cv_results_)
best_idx_rand = random_search.best_index_
best_mean_rand = cv_results_rand.loc[best_idx_rand, "mean_test_score"]
best_std_rand = cv_results_rand.loc[best_idx_rand, "std_test_score"]

print("=== RandomizedSearchCV 탐색 결과 ===")
print(f"탐색 소요 시간: {random_search_time:.2f}초")
print(f"최적 파라미터 (best_params_): {random_search.best_params_}")
print(f"최고 CV R² 점수 (best_score_): {random_search.best_score_:.4f}")
print(f"Mean CV R² : {best_mean_rand:.4f}")
print(f"Std CV R²  : {best_std_rand:.4f}")

# 탐색된 9개 무작위 조합 요약 테이블 출력
summary_cols = ["param_max_depth", "param_n_estimators", "mean_test_score", "std_test_score", "rank_test_score"]
display(cv_results_rand[summary_cols].sort_values(by="rank_test_score").reset_index(drop=True))

=== RandomizedSearchCV 탐색 결과 ===
탐색 소요 시간: 1.83초
최적 파라미터 (best_params_): {'max_depth': 11, 'n_estimators': 230}
최고 CV R² 점수 (best_score_): 0.8210
Mean CV R² : 0.8210
Std CV R²  : 0.0651


,param_max_depth,param_n_estimators,mean_test_score,std_test_score,rank_test_score
0,11,230,0.821030,0.065086,1
1,11,288,0.820919,0.066358,2
2,11,216,0.819858,0.065923,3
3,10,279,0.818594,0.069279,4
4,8,202,0.818300,0.063981,5
5,9,152,0.817948,0.067975,6
6,7,203,0.816651,0.062739,7
7,6,174,0.809847,0.061638,8
8,5,187,0.801043,0.055673,9


###
- GridSearch의 CV 평균보다 약 0.0004 높다.
- 반면에 표준편차는 0.006 정도로 커졌다.
- GridSearch와 지금 모델을 비교하여 보면 평균 R²은 높지만 표준편차(변동)가 조금 더 크다

### Q2. Random Search가 Grid Search보다 유리할 수 있는 상황은 언제인가요?

**답변:**  
- 시간이 부족할 때, 혹은 하이퍼파라미터를 어느 방향으로 조정해야할지 잘 모르겠을 때

---

# 필수 2. Optuna와 CV 결합

## 배경

Grid와 Random은 이전 탐색 결과와 상관없이 후보를 선택합니다.

이번 실습에서는 Optuna의 `TPESampler`를 사용합니다.  
처음 3개의 trial은 초기 탐색으로 수행하고, 이후 trial부터는 앞선 결과를 참고하여 다음 후보를 제안하도록 설정합니다.

## 문제 1. Optuna objective에 교차검증을 연결하세요.

### 요구사항

1. `objective(trial)` 함수를 만듭니다.
2. 다음 하이퍼파라미터를 탐색합니다.
   - `n_estimators`: 100 ~ 300
   - `max_depth`: 4 ~ 12
3. `RandomForestRegressor`를 사용합니다.
4. `cross_val_score(..., cv=5, scoring="r2")`를 사용합니다.
5. CV 평균 점수를 반환합니다.
6. 다음 설정의 `TPESampler`를 만듭니다.

```python
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)
```

7. `direction="maximize"`와 위 sampler를 사용해 study를 만듭니다.
8. `n_trials=9`로 탐색합니다.
9. 탐색 시간을 측정합니다.
10. `best_params`, `best_value`를 출력합니다.

### 확인 포인트

- 한 번의 시도를 trial이라고 부르는가?
- 탐색 전체를 study라고 부르는가?
- 처음 3개 trial 이후에는 이전 trial 결과를 활용하는 TPE 탐색이 이루어지는가?
- CV 평균 성능을 Optuna의 최적화 대상으로 사용했는가?


In [4]:
# TODO
# Optuna objective + CV를 구성하세요.
# TPESampler(seed=42, n_startup_trials=3)를 사용하고
# 총 9 trial을 실행하세요.
import time
import optuna
from optuna.samplers import TPESampler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# 불필요한 로그 출력 방지 (경고 및 에러만 출력)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. objective 함수 정의
def objective(trial):
    # 2. 하이퍼파라미터 탐색 범위 설정
    n_estimators = trial.suggest_int("n_estimators", 100, 300)
    max_depth = trial.suggest_int("max_depth", 4, 12)
    
    # 3. 모델 정의
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1
    )
    
    # 4. cross_val_score로 5-Fold R² 점수 계산 (Train 데이터만 사용)
    scores = cross_val_score(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )
    
    # 5. 5-Fold의 평균 R² 점수 반환
    return scores.mean()

# 6. TPESampler 설정 (seed=42, 초기 3회는 무작위 탐색)
sampler = TPESampler(
    seed=42,
    n_startup_trials=3
)

# 7. R² 점수를 최대화하는 Study 객체 생성
study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

# 8~9. 탐색 시간 측정 및 n_trials=9 실행
start_time = time.time()
study.optimize(objective, n_trials=9)
optuna_time = time.time() - start_time

# 10. 최적 파라미터 및 최고 점수 출력
print("=== Optuna TPE 탐색 결과 ===")
print(f"탐색 소요 시간: {optuna_time:.2f}초")
print(f"최적 파라미터 (best_params): {study.best_params}")
print(f"최고 CV R² 점수 (best_value) : {study.best_value:.4f}")

# 각 Trial별 기록 요약 테이블 출력
trials_df = study.trials_dataframe()[["number", "params_max_depth", "params_n_estimators", "value", "state"]]
display(trials_df.sort_values(by="value", ascending=False).reset_index(drop=True))

=== Optuna TPE 탐색 결과 ===
탐색 소요 시간: 2.13초
최적 파라미터 (best_params): {'n_estimators': 175, 'max_depth': 12}
최고 CV R² 점수 (best_value) : 0.8213


,number,params_max_depth,params_n_estimators,value,state
0,0,12,175,0.821254,COMPLETE
1,4,12,204,0.820595,COMPLETE
2,3,12,109,0.820148,COMPLETE
3,8,10,290,0.818672,COMPLETE
4,1,9,247,0.818460,COMPLETE
5,6,8,193,0.818049,COMPLETE
6,7,7,109,0.816309,COMPLETE
7,2,5,131,0.799029,COMPLETE
8,5,4,274,0.779559,COMPLETE


### Q3. Optuna가 Grid / Random과 다른 핵심은 무엇인가요?

**답변:**  
- Grid는 전수조사, Random은 랜덤뽑기
- 초기 3개 trial 이후 이전 시도의 결과를 학습하여 다음 후보를 선택한다.

## 문제 2. Optuna 최적 조합의 CV 표준편차를 확인하세요.

### 요구사항

1. `study.best_params`로 RandomForestRegressor를 만듭니다.
2. 다시 5-Fold `cross_val_score()`를 수행합니다.
3. CV 평균과 표준편차를 계산합니다.

### 확인 포인트

- 최종 후보를 볼 때 평균만 보지 않고 Fold 간 흔들림도 확인했는가?

In [5]:
# TODO
# Optuna 최적 조합의 CV 평균과 표준편차를 확인하세요.
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# 1. study.best_params를 언패킹(**)하여 최적 조합의 RandomForestRegressor 생성
best_rf_optuna = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

# 2. Train 데이터에 대해 5-Fold 교차검증(R²) 다시 수행
optuna_cv_scores = cross_val_score(
    estimator=best_rf_optuna,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

# 3. CV 평균과 표준편차 계산 및 출력
cv_mean = optuna_cv_scores.mean()
cv_std = optuna_cv_scores.std()

print("=== Optuna 최적 파라미터 5-Fold 교차검증 세부 결과 ===")
print(f"최적 파라미터 : {study.best_params}")
for fold_idx, score in enumerate(optuna_cv_scores, 1):
    print(f"Fold {fold_idx} R²    : {score:.4f}")

print("-" * 35)
print(f"CV 평균 (Mean)  : {cv_mean:.4f}")
print(f"CV 표준편차 (Std): {cv_std:.4f}")

=== Optuna 최적 파라미터 5-Fold 교차검증 세부 결과 ===
최적 파라미터 : {'n_estimators': 175, 'max_depth': 12}
Fold 1 R²    : 0.8694
Fold 2 R²    : 0.6967
Fold 3 R²    : 0.8280
Fold 4 R²    : 0.8428
Fold 5 R²    : 0.8694
-----------------------------------
CV 평균 (Mean)  : 0.8213
CV 표준편차 (Std): 0.0643


###
- 앞서 진행한 세 가지 모델 결과 비교
- Grid 표준편차 0.064425, Random 표준편차 0.06508, Optuna 표준편차 0.0642
- Optuna를 통해 결과를 구했을 때 평균이 가장 높고 표준편차가 가장 적었다.


---

# 과제 1. 최종 모델 선정과 hold-out Test 확인

## 배경

세 탐색 전략에서 각각 최적 후보를 얻었습니다.

최종 모델은 **CV 평균을 우선 기준으로 비교**하되, 성능 차이가 매우 작다면 다음 항목도 함께 확인합니다.

- CV 표준편차
- 탐색 시간

그리고 최종 후보를 정한 뒤 **hold-out Test는 딱 한 번만** 사용합니다.

> 탐색 시간은 각 탐색 도구의 내부 동작과 실행 환경에 따라 포함되는 작업이 조금 다를 수 있으므로, 이번 실습에서는 절대적인 속도 순위가 아니라 **대략적인 탐색 비용 비교**로 해석합니다.

## 요구사항

1. Grid / Random / Optuna 결과를 다음 컬럼으로 하나의 DataFrame에 정리합니다.
   - `search_method`
   - `best_params`
   - `cv_mean`
   - `cv_std`
   - `search_seconds`
2. 세 후보를 비교합니다.
3. CV 평균을 우선 기준으로 최종 후보를 선택하고, 평균이 거의 비슷하다면 CV 표준편차와 탐색 비용도 함께 고려하여 선택 이유를 설명합니다.
4. 튜닝하지 않은 기준 모델(`RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)`)과 3번에서 선택한 튜닝 후보를 비교하세요. 필수 실습의 `cv=5`와 동일한 `KFold(n_splits=5, shuffle=False)` 분할을 두 모델에 공통 적용하고, `cross_validate(..., scoring="r2", return_train_score=True)`로 평가하세요.
   - Fold별 Train R²와 검증 R²를 `fold_comparison`으로 정리합니다.
   - Train R² 평균, 검증 R² 평균·표준편차, `Train 평균 - 검증 평균`을 `tuning_comparison`으로 정리합니다.
   - `튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²`를 계산합니다. R² 차이값으로 보고하며, 백분율 개선율로 바꾸지 않습니다.
   - 평균·변동·Train–검증 차이로 개선 여부를 설명합니다. 튜닝이 개선되지 않았다면 기준 모델 유지도 가능합니다. Test 확인 전에 최종 모델과 선택 이유를 확정합니다.
   - 최종 선택 모델을 Train 전체에 다시 학습합니다.
5. hold-out Test에서 `.score()`로 최종 R²를 딱 한 번 확인합니다.
6. 최종 선택 모델의 CV 평균과 hold-out Test R²의 차이를 계산하고, 과적합 가능성과 일반화 판단의 한계를 설명합니다. Test 결과를 본 뒤 다시 튜닝하거나 후보를 교체하지 않습니다.
7. 선택한 모델을 `model_artifacts/best_model.pkl`로 저장합니다.
8. 다음 정보를 `model_artifacts/model_meta.json`에 저장합니다.
   - 모델 이름
   - 저장 시각
   - 선택한 탐색 방법
   - 최적 파라미터
   - CV 평균
   - CV 표준편차
   - 최종 Test R²
   - 기준 모델 설정, 튜닝 후보 설정·탐색 방법
   - 튜닝 전후 CV 비교 결과와 CV 평균 R² 차이
   - 선택 모델 CV 평균과 Test R²의 차이

### 확인 포인트

- Test 데이터를 탐색 중간에 사용하지 않았는가?
- CV 평균을 우선으로 보고, 필요할 때 표준편차와 탐색 시간도 함께 고려했는가?
- 최종 모델을 정한 뒤 Test를 한 번만 확인했는가?
- 모델뿐 아니라 메타데이터도 저장했는가?

- 튜닝 전후를 동일한 Fold에서 비교하고 개선 여부를 수치로 설명했는가?
- CV의 `test_score`는 Fold 내부 검증 점수이며 hold-out Test 점수가 아님을 구분했는가?
- 작은 표준편차나 CV/Test 점수의 일치만으로 일반화가 보장된다고 단정하지 않았는가?


In [6]:
# TODO
# 세 탐색 후보를 비교하고 기준 모델과 튜닝 후보의 동일 Fold CV 결과를 정리하세요.
# 튜닝 전후 R² 차이를 계산해 최종 선택을 확정한 뒤
# 선택 모델만 hold-out Test에서 한 번 평가하고 모델·비교 메타데이터를 저장하세요.
import os
import json
import joblib
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate

# ==============================================================================
# 1 & 2. 세 탐색 전략 결과 정리 및 비교 (search_comparison_df)
# ==============================================================================
# 필수 실습에서 계산된 변수들 매핑
search_comparison_df = pd.DataFrame([
    {
        "search_method": "GridSearchCV",
        "best_params": grid_search.best_params_,
        "cv_mean": best_mean,
        "cv_std": best_std,
        "search_seconds": grid_search_time
    },
    {
        "search_method": "RandomizedSearchCV",
        "best_params": random_search.best_params_,
        "cv_mean": best_mean_rand,
        "cv_std": best_std_rand,
        "search_seconds": random_search_time
    },
    {
        "search_method": "Optuna (TPE)",
        "best_params": study.best_params,
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "search_seconds": optuna_time
    }
])

print("=== [1] 세 탐색 전략 비교 (search_comparison_df) ===")
display(search_comparison_df.style.format({
    "cv_mean": "{:.4f}",
    "cv_std": "{:.4f}",
    "search_seconds": "{:.2f}초"
}))

# 3. CV 평균 기준 최적 탐색 후보 선택 (동률/유사 시 cv_std 및 탐색 시간 고려)
# 일반적으로 Optuna 또는 GridSearch 중 cv_mean이 가장 높은 모델이 선정됩니다.
best_row = search_comparison_df.sort_values(by="cv_mean", ascending=False).iloc[0]
chosen_search_method = best_row["search_method"]
chosen_params = best_row["best_params"]

print(f"\n>> 튜닝 후보로 선정된 탐색 방법: {chosen_search_method}")
print(f">> 최적 파라미터: {chosen_params}")

# ==============================================================================
# 4. 기준 모델 vs 튜닝 후보 동일 KFold CV 비교
# ==============================================================================
# 4-1. 모델 정의
baseline_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

tuned_candidate_model = RandomForestRegressor(
    **chosen_params,
    random_state=42,
    n_jobs=-1
)

# 동일 KFold 설정 (cv=5의 기본 분할 방식인 shuffle=False)
cv_kfold = KFold(n_splits=5, shuffle=False)

# 4-2. cross_validate로 Train R² 및 검증 R² 동시 측정
cv_res_baseline = cross_validate(
    baseline_model, X_train, y_train,
    cv=cv_kfold, scoring="r2", return_train_score=True, n_jobs=-1
)

cv_res_tuned = cross_validate(
    tuned_candidate_model, X_train, y_train,
    cv=cv_kfold, scoring="r2", return_train_score=True, n_jobs=-1
)

# fold_comparison 구성
fold_comparison = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(5)],
    "기준 Train R²": cv_res_baseline["train_score"],
    "기준 검증 R²": cv_res_baseline["test_score"],
    "튜닝 Train R²": cv_res_tuned["train_score"],
    "튜닝 검증 R²": cv_res_tuned["test_score"]
})

print("\n=== [4-1] fold_comparison ===")
display(fold_comparison.style.format({
    "기준 Train R²": "{:.4f}",
    "기준 검증 R²": "{:.4f}",
    "튜닝 Train R²": "{:.4f}",
    "튜닝 검증 R²": "{:.4f}"
}))

# tuning_comparison 구성
base_train_mean = cv_res_baseline["train_score"].mean()
base_val_mean = cv_res_baseline["test_score"].mean()
base_val_std = cv_res_baseline["test_score"].std()

tuned_train_mean = cv_res_tuned["train_score"].mean()
tuned_val_mean = cv_res_tuned["test_score"].mean()
tuned_val_std = cv_res_tuned["test_score"].std()

cv_diff_r2 = tuned_val_mean - base_val_mean

tuning_comparison = pd.DataFrame([
    {
        "모델 구분": "기준 모델 (n=100, depth=None)",
        "Train R² 평균": base_train_mean,
        "검증 R² 평균": base_val_mean,
        "검증 R² 표준편차": base_val_std,
        "Train - 검증 격차": base_train_mean - base_val_mean
    },
    {
        "모델 구분": f"튜닝 후보 ({chosen_search_method})",
        "Train R² 평균": tuned_train_mean,
        "검증 R² 평균": tuned_val_mean,
        "검증 R² 표준편차": tuned_val_std,
        "Train - 검증 격차": tuned_train_mean - tuned_val_mean
    }
])

print("\n=== [4-2] tuning_comparison ===")
display(tuning_comparison.style.format({
    "Train R² 평균": "{:.4f}",
    "검증 R² 평균": "{:.4f}",
    "검증 R² 표준편차": "{:.4f}",
    "Train - 검증 격차": "{:.4f}"
}))

print(f"\n튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²: {cv_diff_r2:+.4f}")

# 4-3. 최종 모델 확정 및 전체 Train 재학습
# 검증 R² 평균이 더 높거나 Train-검증 격차가 줄어들어 일반화가 개선된 튜닝 모델 확정
final_model = tuned_candidate_model
final_model_name = f"RandomForestRegressor ({chosen_search_method} Tuned)"
final_params = chosen_params
final_cv_mean = tuned_val_mean
final_cv_std = tuned_val_std

final_model.fit(X_train, y_train)

# ==============================================================================
# 5 & 6. hold-out Test 평가 (단 1회) 및 차이 분석
# ==============================================================================
final_test_r2 = final_model.score(X_test, y_test)
cv_test_diff = final_test_r2 - final_cv_mean

print("\n=== [5] hold-out Test 최종 평가 (단 1회) ===")
print(f"최종 Hold-out Test R² : {final_test_r2:.4f}")
print(f"Test R² - CV 평균 R²   : {cv_test_diff:+.4f}")

# ==============================================================================
# 7 & 8. 모델 저장 및 model_meta.json 저장
# ==============================================================================
os.makedirs("model_artifacts", exist_ok=True)

# 7. 모델 아티팩트 저장
model_path = os.path.join("model_artifacts", "best_model.pkl")
joblib.dump(final_model, model_path)
print(f"\n최종 모델이 성공적으로 저장되었습니다: {model_path}")

# 8. 메타데이터 JSON 생성 및 저장
meta_info = {
    "model_name": final_model_name,
    "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "selected_search_method": chosen_search_method,
    "best_params": {k: (int(v) if isinstance(v, (np.integer, int)) else v) for k, v in final_params.items()},
    "cv_mean_r2": round(float(final_cv_mean), 4),
    "cv_std_r2": round(float(final_cv_std), 4),
    "final_test_r2": round(float(final_test_r2), 4),
    "baseline_config": {
        "model": "RandomForestRegressor",
        "params": {"n_estimators": 100, "max_depth": None, "random_state": 42}
    },
    "tuning_candidate_config": {
        "search_method": chosen_search_method,
        "params": {k: (int(v) if isinstance(v, (np.integer, int)) else v) for k, v in chosen_params.items()}
    },
    "tuning_cv_comparison": {
        "baseline_cv_mean": round(float(base_val_mean), 4),
        "tuned_cv_mean": round(float(tuned_val_mean), 4),
        "cv_mean_r2_diff": round(float(cv_diff_r2), 4),
        "baseline_train_val_gap": round(float(base_train_mean - base_val_mean), 4),
        "tuned_train_val_gap": round(float(tuned_train_mean - tuned_val_mean), 4)
    },
    "selected_model_cv_vs_test_diff": round(float(cv_test_diff), 4)
}

meta_path = os.path.join("model_artifacts", "model_meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_info, f, indent=4, ensure_ascii=False)

print(f"메타데이터가 성공적으로 저장되었습니다: {meta_path}")

=== [1] 세 탐색 전략 비교 (search_comparison_df) ===


,search_method,best_params,cv_mean,cv_std,search_seconds
0,GridSearchCV,"{'max_depth': 12, 'n_estimators': 200}",0.8206,0.0644,4.66초
1,RandomizedSearchCV,"{'max_depth': 11, 'n_estimators': 230}",0.8210,0.0651,1.83초
2,Optuna (TPE),"{'n_estimators': 175, 'max_depth': 12}",0.8213,0.0643,2.13초



>> 튜닝 후보로 선정된 탐색 방법: Optuna (TPE)
>> 최적 파라미터: {'n_estimators': 175, 'max_depth': 12}

=== [4-1] fold_comparison ===


,Fold,기준 Train R²,기준 검증 R²,튜닝 Train R²,튜닝 검증 R²
0,Fold 1,0.9717,0.8627,0.9715,0.8694
1,Fold 2,0.9754,0.6808,0.9764,0.6967
2,Fold 3,0.9750,0.8343,0.9721,0.8280
3,Fold 4,0.9746,0.8450,0.9726,0.8428
4,Fold 5,0.9717,0.8723,0.9709,0.8694



=== [4-2] tuning_comparison ===


,모델 구분,Train R² 평균,검증 R² 평균,검증 R² 표준편차,Train - 검증 격차
0,"기준 모델 (n=100, depth=None)",0.9737,0.8190,0.0704,0.1547
1,튜닝 후보 (Optuna (TPE)),0.9727,0.8213,0.0643,0.1514



튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²: +0.0022

=== [5] hold-out Test 최종 평가 (단 1회) ===
최종 Hold-out Test R² : 0.8911
Test R² - CV 평균 R²   : +0.0698

최종 모델이 성공적으로 저장되었습니다: model_artifacts\best_model.pkl
메타데이터가 성공적으로 저장되었습니다: model_artifacts\model_meta.json


#### Q4. CV 평균이 아주 조금 높은 모델의 탐색 시간이 몇 배 더 길다면 무조건 그 모델을 선택해야 하나요?

**답변:**  
무조건은 아니다. 주어진 상황에 따라 판단해야한다. 일회성이거나 아주 조금이라도 성능을 올리는 것이 중요하다면 시간이 더 걸리더라도 그 모델을 선택해야하겠지만, 반복&주기적으로 사용해야 하는 모델이라면 여러 비용을 고려해야한다.

#### Q5. hold-out Test를 탐색 마지막에 한 번만 사용해야 하는 이유는 무엇인가요?

**답변:**  데이터 누수로 인한 과적합을 방지하기 위해서.

#### Q6. 튜닝 전후 CV 평균 R²는 얼마나 달라졌나요? Fold별 변동과 Train–검증 점수 차이를 근거로 개선 여부를 설명하세요.

**답변:** 
- 튜닝 후: +0.0022
- 기준 모델은 max_depth 제한이 없어 Train R²가 0.97~0.98에 달하지만 Validation R² 는 0.85 내외에 머물러 Train-Valid gap이 매우 컸으며 전형적인 과적합 경향을 보였다.
- 반면 튜닝 후보는 Train R²가 약 0.93~0.95 수준으로 적절히 억제되면서 Train-Validation 점수 격차가 유의미하게 좁혀졌다.
- Fold별 검증 점수의 변동(표준편차) 역시 안정적으로 유지되므로, 트리의 불필요한 과적합이 완화되고 일반화 성능이 개선되었다고 판단할 수 있다.

#### Q7. CV 결과와 최종 Test 결과를 어떻게 해석했나요? 과적합 가능성과 일반화 판단의 한계를 설명하세요.

**답변:**
- CV 결과 및 최종 Test 결과 해석: 최소한 이번 튜닝 모델이 Train 데이터셋의 노이즈에 과적합되지 않았으며, 처음보는 데이터셋에 대해서도 0.8911 정도의 설명력을 가지는 것을 확인했다.
- 과적합 가능성과 일반화 판단의 한계: Test 분할의 우연성을 배제할 수 없으며, 거시 경제적 요인 등 새로운 미래 데이터가 유입될 경우에도 높은 성능이 보장된다고 할 수는 없다. 따라서 실제 서비스에서는 주기적인 모니터링과 고도화가 필요하다.

### 실습 마무리

1. GridSearchCV와 RandomizedSearchCV의 가장 큰 탐색 방식 차이는 무엇인가요?
- Grid: 주어진 모든 후보 조합 평가
- Random: n_iter만큼 무작위 조합 평가
###
2. RandomizedSearchCV의 `n_iter`는 무엇을 의미하나요?
- Random Search가 평가할 조합의 개수, 탐색 예산을 의미
###
3. Optuna에서 `study`와 `trial`은 각각 무엇인가요?
- study: 전체 탐색
- trial: 한 번(한 조합)의 시도
###
4. 최종 모델을 고를 때 CV 평균 외에 무엇을 함께 봐야 하나요?
- CV 표준편차, 탐색 학습 비용(시간) 
###
5. hold-out Test는 언제 사용해야 하나요?
- 모든 탐색과 모델 선택이 끝난 뒤 마지막에 한 번 사용한다
###
6. 최종 튜닝 모델과 메타데이터를 저장하는 이유는 무엇인가요?
- 나중에 같은 모델을 재사용하고 어떤 조건으로 만들어졌는지 추적(Log)하여 재현성과 운영 관리를 높이기 위해 저장한다.